# Infrared (DRIFTS): Thermo OMNIC `.spa`

A short series of DRIFTS spectra exported by OMNIC, one `.spa` file per scan.
`ldp.IRData` is the reader namespace; what it returns is a `SpectroscopyData`,
the one class for infrared and Raman data. The example files are the test
fixtures under `tests/spectroscopy/infrared/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
DATA = Path("..") / "tests" / "spectroscopy" / "infrared" / "omnic-test-data"
sorted(p.name for p in DATA.glob("*.spa"))

## Read a directory

A directory reads into one 2-D object `(scan, wavenumber)`. The time of every
scan comes from the acquisition timestamp stored inside each file, not from the
rounded hours in the file name. `tos_start` anchors the elapsed time `tos`
(seconds) to the wall clock; a timezone-aware `pandas.Timestamp` is safest.
`backend="omnic"` uses the built-in parser (the default `"auto"` uses
SpectroChemPy when it is installed).

In [ ]:
tos_start = pd.Timestamp("2026-03-26 14:40:00", tz="Europe/Amsterdam")
ir_raw = ldp.IRData.from_omnic_spa(DATA, tos_start=tos_start, backend="omnic")
ir_raw

In [ ]:
print("data type:", ir_raw.data_type)
print("technique:", ir_raw.technique)
print("number of scans:", ir_raw.shape[0])
print("tos / s:", ir_raw.tos)
print("first timestamp:", ir_raw.timestamps[0])

## The axis in any unit

The axis is stored once, in m^-1. `axis(unit)` reads it in any spectral unit
and `axis_label(unit)` gives the matching plot label.

In [ ]:
for unit in ("cm^-1", "um", "eV"):
    axis = ir_raw.axis(unit)
    print(f"{ir_raw.axis_label(unit):28s} {axis.min():10.4g} to {axis.max():10.4g}")

## A processing chain

Every method returns a new object, so the raw data stays available. Here: cut
the mid-infrared window, subtract a baseline (offset from an anchor window plus
a PCHIP curve through control points) and smooth.

In [ ]:
ir = (
    ir_raw
    .select_wavenumber_range(1000, 3900)
    .correct_baseline(anchor_range_cm=(2600, 2500), control_points_cm=[3900, 3450, 2720, 2500, 1850, 1000])
    .smooth_savgol(window_length=11, polyorder=3)
)
print(ir.format_history())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for i in range(ir.shape[0]):
    ax.plot(ir.wavenumber_per_cm, ir.values[i], lw=0.8, color=plt.cm.viridis(i / ir.shape[0]))
ax.invert_xaxis()
ax.set_xlabel(ir.axis_label("cm^-1"))
ax.set_ylabel(f"{ir.data_type} / –")
ax.set_title("Baseline-corrected DRIFTS spectra")
plt.show()

## Band intensity over time

`get_evolution` returns an `xarray.DataArray` along `scan`, with `tos` as a
coordinate.

In [ ]:
evolution = ir.get_evolution(2350)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(evolution["tos"], evolution, "o-")
ax.set_xlabel("time on stream / s")
ax.set_ylabel(f"{ir.data_type} at 2350 cm^-1 / –")
plt.show()

## Save and reload

`to_netcdf` keeps data, coordinates, metadata and the processing history.
Processed data never goes next to the raw files; here a temporary folder is used.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = Path(folder) / "ir.nc"
    ir.to_netcdf(path)
    reloaded = ldp.SpectroscopyData.from_netcdf(path)

np.testing.assert_allclose(reloaded.values, ir.values)
print(reloaded.format_history())